# Train models

In [1]:
import sys
working_directory = "/home/icb/kemal.inecik/work/codes/sctram"
sys.path.append(working_directory)

import os
import itertools
import subprocess
import time

In [2]:
dataset_dir = "/home/icb/kemal.inecik/lustre_workspace/temp_sctram_data"
helpers_directory = os.path.join(os.getcwd(), "helper")
logs_directory = os.path.join(os.getcwd(), "logs")

In [3]:
def is_job_running(job_name):
    result = subprocess.run(['squeue', '-o', '%.28i %.28j %.28u %R', '-u', 'kemal.inecik'],  capture_output=True, text=True)
    jobs = [[j.strip() for j in i.split()]for i in result.stdout.strip().split('\n')]
    for jobid, jobname, jobuser, jobnode in jobs:
        if job_name == jobname:
            return True
    return False

```
#SBATCH -J {job_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

```
#SBATCH -J {job_name}
#SBATCH -p cpu_p
#SBATCH --qos cpu_normal
#SBATCH -c 32
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 1-23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}
```

In [5]:
job_count = 0
# epochs = list(itertools.chain(range(0, 100), range(100, 600, 5), range(600, 1100, 10)))
epochs = list(itertools.chain(range(0, 19), range(19, 52, 3), range(52, 100, 6), range(100, 200, 10), range(200, 400, 20), range(400, 1001, 40)))
overwrite = False
print(f" - Number of models to be trained: {len(epochs)!r}")

cpu_gpu = "gpu"
for epoch in epochs:

    for model_str in ["scanvi", "scvi"]:
        
        output_dir_path = os.path.join(dataset_dir, f"model_suo_incremental_training_{model_str}_epoch_{epoch}")
        log_file = os.path.join(logs_directory, f"slurm_out_model_suo_incremental_training_{cpu_gpu}_{model_str}_epoch_{epoch}.log")
        job_name = f"incr_{cpu_gpu}_{model_str}_{epoch}"
        python_name = f"model_training.py"

        if is_job_running(job_name):
            print(f"Training {job_name!r} on {cpu_gpu!r} keeps going for model {model_str!r} and for epoch {epoch!r}.")
        elif overwrite or not os.path.exists(output_dir_path) or not os.path.isdir(output_dir_path):
            try:
                slurm_script = f"""#!/bin/bash
#SBATCH -J {job_name}
#SBATCH -p gpu_p
#SBATCH --qos=gpu_normal
#SBATCH --gres=gpu:1
#SBATCH -c 6
#SBATCH --mem=159G
#SBATCH --nice=0
#SBATCH -t 23:50:00
#SBATCH -o {log_file}
#SBATCH -e {log_file}

source activate sctram_dev_env
python -u {os.path.join(helpers_directory, python_name)} --epoch "{epoch}" --model_str "{model_str}" --overwrite "{overwrite}"
    """
                script_name = os.path.join(logs_directory, f"slurm_job_model_suo_incremental_training_{cpu_gpu}_{model_str}_epoch_{epoch}.sh")
                with open(script_name, "w") as f:
                    f.write(slurm_script)

                print(f"Submitted job on {cpu_gpu!r} {job_count+1} on {cpu_gpu!r}: {job_name!r} for model {model_str!r} and for epoch {epoch!r}")
                subprocess.run(["sbatch", script_name])
                job_count += 1
            finally:
                time.sleep(0.05)
                # os.remove(script_name)

            assert is_job_running(job_name), f"Training {job_name!r} on {cpu_gpu!r} error: model {model_str!r} and for epoch {epoch!r}."
        else:
            print(f"Model exists for model {model_str!r} and for epoch {epoch!r}")
    # if job_count > 32:
    #     break

print(f" - Number of jobs submitted: {job_count}")

 - Number of models to be trained: 74
Submitted job on 'gpu' 1 on 'gpu': 'incr_gpu_scanvi_0' for model 'scanvi' and for epoch 0
Submitted batch job 33740631
Submitted job on 'gpu' 2 on 'gpu': 'incr_gpu_scvi_0' for model 'scvi' and for epoch 0
Submitted batch job 33740632
Submitted job on 'gpu' 3 on 'gpu': 'incr_gpu_scanvi_1' for model 'scanvi' and for epoch 1
Submitted batch job 33740633
Submitted job on 'gpu' 4 on 'gpu': 'incr_gpu_scvi_1' for model 'scvi' and for epoch 1
Submitted batch job 33740634
Submitted job on 'gpu' 5 on 'gpu': 'incr_gpu_scanvi_2' for model 'scanvi' and for epoch 2
Submitted batch job 33740635
Submitted job on 'gpu' 6 on 'gpu': 'incr_gpu_scvi_2' for model 'scvi' and for epoch 2
Submitted batch job 33740636
Submitted job on 'gpu' 7 on 'gpu': 'incr_gpu_scanvi_3' for model 'scanvi' and for epoch 3
Submitted batch job 33740637
Submitted job on 'gpu' 8 on 'gpu': 'incr_gpu_scvi_3' for model 'scvi' and for epoch 3
Submitted batch job 33740638
Submitted job on 'gpu' 9 o